In [ ]:
#installs fredapi
!pip install fredapi
#installs scikit-learn
!pip install scikit-learn

#imports pandas toolset
import pandas as pd
#Imports the FRED tool from the fredapi package
from fredapi import Fred
#imports plotting toolset
import matplotlib.pyplot as plt
#imports statistics toolset
import statistics
#imports the logistic regression tool from scikit-learn
from sklearn.linear_model import LogisticRegression

#Authenticates using API key
fred = Fred(api_key='Insert API Key*')

#pulls the ten year treasury yield series from the API and assigns it to the variable "ten_year"
ten_year = fred.get_series('DGS10')
#pulls the two year treasury yield series from the API and assigns it to the variable "two_year"
two_year = fred.get_series('DGS2')
#pulls NBER's official US recession indicator (1 = recession, 0 = not a recession)
recession = fred.get_series('USREC')

#filters ten year series so it begins at the same time as the two year series
ten_year = ten_year[ten_year.index >= two_year.index.min()]
#filters recession series so it begins at the same time as the two year series
recession = recession[recession.index >= two_year.index.min()]

#plots the ten-year and two-year yields together on a line chart
plt.plot(ten_year, label='10-Year Yield')
plt.plot(two_year, label='2-Year Yield')
#shades the background when recession = 1
plt.fill_between(recession.index, 0, 18, where=(recession == 1), color='grey', alpha=0.3, label='Recession')
#titles the chart and the axes
plt.title('2-Year vs 10-Year Treasury Yields, with recessions shaded')
plt.xlabel('Date')
plt.ylabel('Yield (%)')
#adds a legend to the chart
plt.legend()
#shows chart
plt.show()

#converts daily yields to monthly averages, to match recession's monthly frequency and smooth out short-term flickers
ten_year_monthly = ten_year.resample('ME').mean()
two_year_monthly = two_year.resample('ME').mean()

#creates a new series based on when the 2 year yield exceeds the ten year
inverted = two_year_monthly > ten_year_monthly
#shifts the inverted series forward by one day so each day can be compared with the previous one
previous_day = inverted.shift(1, fill_value=False)
#finds the exact days that inversions started
inversion_start = inverted & (~previous_day)

#gets the actual dates where inversions started
inversion_dates = inverted[inversion_start].index
#gets the actual dates where recessions started
recession_started = (recession == 1) & (recession.shift(1).fillna(0) == 0)
recession_dates = recession[recession_started].index
#finds months where a recession ended (this month is 0, previous month was 1)
recession_ended = (recession == 0) & (recession.shift(1).fillna(0) == 1)
recession_end_dates = recession[recession_ended].index

#checks whether each inversion was already in a recession when it happened
valid_inversions = []
for inv_date in inversion_dates:
    inside_recession = False
    for start, end in zip(recession_dates, recession_end_dates):
        if start <= inv_date <= end:
            inside_recession = True
    if not inside_recession:
        valid_inversions.append(inv_date)

#for every inversion date finds the recession date that comes after it
lead_times = []
for inv_date in valid_inversions:
    future_recessions = recession_dates[recession_dates > inv_date]
    if len(future_recessions) > 0:
        next_recession = future_recessions[0]
        months_lag = (next_recession.year - inv_date.year) * 12 + (next_recession.month - inv_date.month)
        lead_times.append((inv_date, next_recession, months_lag))
    else:
        lead_times.append((inv_date, None, None))

#pulls out just the numeric lead times between inverson and a recession
valid_lags = [lag for (inv, rec, lag) in lead_times if lag is not None]
#prints summary statistics for lead time
print("Min lead time:", min(valid_lags))
print("Max lead time:", max(valid_lags))
print("Average lead time:", sum(valid_lags) / len(valid_lags))
print("Standard deviation:", statistics.stdev(valid_lags))

#calculates the 10Y/2Y spread for every month (positive = normal curve, negative = inverted)
spread = ten_year_monthly - two_year_monthly

#for each month, checks whether a recession started within the next 12 months
recession_within_12m = []

for date in spread.index:
    # calculates the date exactly 12 months after this one
    window_end = date + pd.DateOffset(months=12)
    future_recessions_in_window = recession_dates[(recession_dates > date) & (recession_dates <= window_end)]
    recession_within_12m.append(1 if len(future_recessions_in_window) > 0 else 0)

#combines spread values and recession labels into one dataframe
training_data = pd.DataFrame({
    'spread': spread,
    'recession_within_12m': recession_within_12m
})

#drop the most recent 12 months for obvious reasons
training_data = training_data.iloc[:-12]

training_data.tail()

#separate the input (spread) from the label (recession_within_12m)
X = training_data[['spread']]
y = training_data['recession_within_12m']

#creates and fit the model
model = LogisticRegression()
model.fit(X, y)

#get the model's probability estimate for a recession within 12 months, using the current spread
current_spread_df = pd.DataFrame({'spread': [spread.iloc[-1]]})
probability = model.predict_proba(current_spread_df)

current_probability = probability[0][1]
print(f"Current recession probability: {current_probability:.1%}")
print(f"Historical range preceding actual recessions: 18.5% - 51.4%")

#The 18%/35% cutoffs should be read as directionally informative, not statistically rigorous boundaries.
if current_probability < 0.18:
if current_probability < 0.18:
    print("Below the lowest reading that has historically preceded a recession in this dataset — no tactical deviation from strategic allocation is warranted.")
elif current_probability < 0.35:
    print("Within the range that has historically preceded a recession — consider a modest tactical tilt: reduce cyclical equity exposure, extend duration, trim high-yield credit exposure.")
else:
    print("Above most historical pre-recession readings, approaching levels seen only before the most severe downturn (1981-82) — a meaningful tactical tilt is warranted: underweight cyclicals, extend duration further, reduce credit exposure, increase cash/short-duration allocation.")